In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# PIPELINE DESCRIPTION
The pipeline performs median imputation of clinical variables with missingness indicators, encodes categorical features, and standardizes continuous inputs. SNPs are filtered, standardized, clustered using hierarchical clustering, and selected per cluster via weighted logistic regression, with priority SNPs preserved. A polygenic risk score (PRS) for MACE is then constructed from the selected SNPs. Final models include a penalized logistic regression combining clinical variables, genetic status, and PRS for MACE prediction, and a Gaussian Naive Bayes model using age and genetic status for severity prediction. Predictions are exported for submission.

# PREPROCESSING

Clinical features are median-imputed with additional missingness indicators, while remaining missing values are set to zero. Categorical variables are integer-encoded based on training data. Continuous clinical features are standardized before modeling. Also the SNPs are set up (there are two categories of SNPs).

## Loading data

In [2]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

## Imputation

In [3]:
clinical_cols = ['Age_Baseline', 'Age_Diag', 'BMI', 'BSA', 'Epaiss_max', 'Diam_OG']

for col in clinical_cols:
    if col in df_train.columns:
        df_train[col + "_missing"] = df_train[col].isna().astype(int)
        df_test[col + "_missing"]  = df_test[col].isna().astype(int)
        med = df_train[col].median()
        df_train[col] = df_train[col].fillna(med)
        df_test[col]  = df_test[col].fillna(med)

df_train = df_train.fillna(0)
df_test  = df_test.fillna(0)

## Categorical Encoding

In [4]:
cat_cols = ['Genre', 'Gradient', 'TVNS', 'FEVG', 'ATCD_MS', 'SYNCOPE', 'Variant.Pathogene']

for col in cat_cols:
    if col in df_train.columns:
        mapping = {v: i for i, v in enumerate(df_train[col].unique())}
        df_train[col] = df_train[col].map(mapping)
        df_test[col]  = df_test[col].map(mapping).fillna(-1)

baseline_clinics = [
    'Age_Baseline', 'Age_Diag', 'BMI', 'BSA', 'Genre', 'Gradient',
    'Epaiss_max', 'TVNS', 'FEVG', 'ATCD_MS', 'SYNCOPE', 'Diam_OG'
]

mendelian = ['Variant.Pathogene']

corr_mace = df_train[baseline_clinics + mendelian + ['OUTCOME MACE']].corr()['OUTCOME MACE']
print(corr_mace)

Age_Baseline         0.183682
Age_Diag             0.029867
BMI                  0.096773
BSA                  0.040934
Genre                0.042987
Gradient            -0.002625
Epaiss_max           0.107563
TVNS                 0.089771
FEVG                 0.197496
ATCD_MS              0.116181
SYNCOPE             -0.017773
Diam_OG              0.146316
Variant.Pathogene    0.102240
OUTCOME MACE         1.000000
Name: OUTCOME MACE, dtype: float64


## SNPs Setup

In [5]:
all_snps = [f"SNP{i}" for i in range(1, 289)]
priority_snps = [f"SNP{i}" for i in range(1, 76)]
usable_snps = [s for s in all_snps if s in df_train.columns and df_train[s].nunique() > 1]

# OUTCOME MACE PREDICTION

## Feature Selection

### SNPs Selection + PRS

Standardized SNPs are hierarchically clustered using a correlation-based distance, and the most informative SNPs are selected per cluster using weighted univariate logistic regression, with the priority SNPs being preserved. The number of SNP clusters is optimized by grid search, and cluster-wise selected SNPs are globally ranked using logistic regression weights to retain a final set of up to 50 variants while preserving the priority SNPs. A polygenic risk score (PRS) for MACE is then computed as a weighted sum of the selected SNPs, with weights derived from univariate balanced logistic regression coefficients.

#### Selecting SNPs with Agglomerative Clustering

In [6]:
X_snps_std = (df_train[usable_snps] - df_train[usable_snps].mean()) / df_train[usable_snps].std()
corr = X_snps_std.corr().fillna(0)
dist = 1 - np.abs(corr)
Z = linkage(squareform(dist.values, checks=False), method="average")


def select_top_snps(snps, max_snps=50, priority_list=priority_snps):
    weights = {}
    for snp in snps:
        lr = LogisticRegression(max_iter=500, class_weight="balanced", solver="lbfgs")
        lr.fit(df_train[[snp]], df_train["OUTCOME MACE"])
        weights[snp] = abs(lr.coef_[0][0])
    
    snps_priority = [s for s in snps if s in priority_list]
    snps_sorted = sorted(snps, key=lambda s: weights[s], reverse=True)
    
    final_snps = snps_priority[:]
    for s in snps_sorted:
        if s not in final_snps:
            final_snps.append(s)
        if len(final_snps) >= max_snps:
            break
    return final_snps, weights

#### Optimal K clusters 

In [7]:
# OPTIMAL K (With Weighted Sorting)
K_grid = [5, 8, 10, 12, 15, 18, 20]
best_K = K_grid[0]
selected_snps_final = []

# Pre-calculate global weights to use for the "Final Cut" ranking
global_weights = {}
for snp in usable_snps:
    lr = LogisticRegression(class_weight="balanced").fit(df_train[[snp]], df_train["OUTCOME MACE"])
    global_weights[snp] = abs(lr.coef_[0][0])

for K in K_grid:
    labels = fcluster(Z, t=K, criterion="maxclust")
    clusters = {}
    for snp, lab in zip(usable_snps, labels):
        clusters.setdefault(lab, []).append(snp)
    
    temp_pool = []
    for cl, snps in clusters.items():
        snps_sel, _ = select_top_snps(snps, max_snps=50)
        temp_pool.extend(snps_sel)
    
    temp_pool = list(dict.fromkeys(temp_pool)) # Unique candidate pool
    
    if len(temp_pool) <= 50:
        selected_snps_final = temp_pool
        best_K = K
        break
    else:
        # THE BETTER IDEA: Sort the pool by weight before cutting to 50
        best_K = K
        # Priority Shield: Ensure priority SNPs stay, then fill rest with top weights
        p_in_pool = [s for s in temp_pool if s in priority_snps]
        non_p_in_pool = [s for s in temp_pool if s not in priority_snps]
        
        sorted_non_p = sorted(non_p_in_pool, key=lambda s: global_weights[s], reverse=True)
        selected_snps_final = (p_in_pool + sorted_non_p)[:50]
        break

print(f"K optimal = {best_K} | total SNPs = {len(selected_snps_final)}")
print(f"Selected List: {selected_snps_final}")

K optimal = 5 | total SNPs = 50
Selected List: ['SNP1', 'SNP2', 'SNP7', 'SNP15', 'SNP18', 'SNP19', 'SNP20', 'SNP21', 'SNP24', 'SNP25', 'SNP27', 'SNP28', 'SNP35', 'SNP36', 'SNP37', 'SNP38', 'SNP39', 'SNP42', 'SNP44', 'SNP46', 'SNP49', 'SNP50', 'SNP51', 'SNP59', 'SNP60', 'SNP67', 'SNP71', 'SNP73', 'SNP74', 'SNP75', 'SNP3', 'SNP23', 'SNP29', 'SNP45', 'SNP58', 'SNP4', 'SNP5', 'SNP6', 'SNP8', 'SNP9', 'SNP10', 'SNP12', 'SNP16', 'SNP22', 'SNP26', 'SNP30', 'SNP32', 'SNP53', 'SNP54', 'SNP56']


#### Computing PRS

In [8]:
weights_all = {s: LogisticRegression(class_weight="balanced").fit(df_train[[s]], df_train["OUTCOME MACE"]).coef_[0][0] for s in selected_snps_final}
df_train["PRS_MACE"] = df_train[selected_snps_final].dot(pd.Series(weights_all))
df_test["PRS_MACE"] = df_test[selected_snps_final].dot(pd.Series(weights_all))

### Baseline features Selection

In [10]:
corr_mace = df_train[baseline_clinics + mendelian + ['OUTCOME MACE']].corr()['OUTCOME MACE']
print(corr_mace) # Variables with low corr with MACE were removed

df_train['Age_Variant_Fused'] = df_train['Age_Baseline'] * df_train['Variant.Pathogene']
df_test['Age_Variant_Fused'] = df_test['Age_Baseline'] * df_test['Variant.Pathogene']

if "SYNCOPE" in baseline_clinics:
    baseline_clinics.remove("SYNCOPE")
if "Gradient" in baseline_clinics :
    baseline_clinics.remove("Gradient")
if "Age_Variant_Fused" not in baseline_clinics:
    baseline_clinics.append("Age_Variant_Fused")

print(df_train[["Age_Variant_Fused"] + ['OUTCOME MACE']].corr()['OUTCOME MACE'])

Age_Baseline         0.183682
Age_Diag             0.029867
BMI                  0.096773
BSA                  0.040934
Genre                0.042987
Epaiss_max           0.107563
TVNS                 0.089771
FEVG                 0.197496
ATCD_MS              0.116181
Diam_OG              0.146316
Age_Variant_Fused    0.138332
Variant.Pathogene    0.102240
OUTCOME MACE         1.000000
Name: OUTCOME MACE, dtype: float64
Age_Variant_Fused    0.138332
OUTCOME MACE         1.000000
Name: OUTCOME MACE, dtype: float64


## Fit and predict

In [11]:
scaler = StandardScaler()
X_tr = np.hstack([scaler.fit_transform(df_train[baseline_clinics]), df_train[mendelian + ["PRS_MACE"]].values])
X_te = np.hstack([scaler.transform(df_test[baseline_clinics]), df_test[mendelian + ["PRS_MACE"]].values])

model_mace = LogisticRegression(class_weight="balanced", C=0.01, solver="lbfgs", max_iter=1000).fit(X_tr, df_train["OUTCOME MACE"])

# OUTCOME SEVERITY

In [12]:
model_sev = GaussianNB()
baseline_clinics = ['Age_Baseline', 'Age_Variant_Fused'] # Genre barely changes anything in the model.
model_sev.fit(df_train[baseline_clinics + mendelian], df_train['OUTCOME SEVERITY'])
y_probs_final = model_sev.predict_proba(df_test[baseline_clinics + mendelian])[:, 1]

# RESULTS

## Cross-Validation

In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# Setup Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metrics = {
    "MACE": {"f1": [], "auc": []},
    "SEVERITY": {"auc": []} # Severity is often treated as a ranking/probabilistic task
}

# We iterate through folds to simulate how the model performs on unseen data
for train_idx, val_idx in skf.split(df_train, df_train["OUTCOME MACE"]):
    # Split
    X_fold_train, X_fold_val = X_tr[train_idx], X_tr[val_idx]
    y_mace_train, y_mace_val = df_train["OUTCOME MACE"].iloc[train_idx], df_train["OUTCOME MACE"].iloc[val_idx]
    
    y_sev_train, y_sev_val = df_train["OUTCOME SEVERITY"].iloc[train_idx], df_train["OUTCOME SEVERITY"].iloc[val_idx]
    X_sev_train = df_train[baseline_clinics + mendelian].iloc[train_idx]
    X_sev_val = df_train[baseline_clinics + mendelian].iloc[val_idx]

    # --- MACE Metrics ---
    model_mace.fit(X_fold_train, y_mace_train)
    mace_preds = model_mace.predict(X_fold_val)
    mace_probs = model_mace.predict_proba(X_fold_val)
    
    metrics["MACE"]["f1"].append(f1_score(y_mace_val, mace_preds, average='weighted'))
    metrics["MACE"]["auc"].append(roc_auc_score(y_mace_val, mace_probs, multi_class='ovr'))

    # --- SEVERITY Metrics ---
    model_sev.fit(X_sev_train, y_sev_train)
    sev_probs = model_sev.predict_proba(X_sev_val)[:, 1]
    metrics["SEVERITY"]["auc"].append(roc_auc_score(y_sev_val, sev_probs))

print("\n--- Cross-Validation Results ---")
print(f"MACE F1-Score: {np.mean(metrics['MACE']['f1']):.4f} (+/- {np.std(metrics['MACE']['f1']):.3f})")
print(f"MACE AUC:      {np.mean(metrics['MACE']['auc']):.4f}")
print(f"SEV AUC:       {np.mean(metrics['SEVERITY']['auc']):.4f}")


--- Cross-Validation Results ---
MACE F1-Score: 0.6203 (+/- 0.035)
MACE AUC:      0.6462
SEV AUC:       0.6319


## Submission file

In [ ]:
submission = pd.DataFrame({
    "trustii_id": df_test["trustii_id"],
    "OUTCOME MACE": model_mace.predict(X_te).astype(int),
    "OUTCOME SEVERITY": y_probs_final
})
submission.to_csv("Final_Output.csv", index=False)